In [1]:
!pip install -q transformers datasets evaluate fvcore accelerate thop

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Đang chạy trên thiết bị: {device}")

Đang chạy trên thiết bị: cuda


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VanillaKDLoss(nn.Module):
    """
    Vanilla Knowledge Distillation (Hinton)
    """
    def __init__(self, temperature=4.0):
        super(VanillaKDLoss, self).__init__()
        self.T = temperature
        self.kl_div = nn.KLDivLoss(reduction='none')

    def forward(self, logits_student, logits_teacher, target_masks):
        h, w = target_masks.shape[1], target_masks.shape[2]
        logits_student = F.interpolate(logits_student, size=(h, w), mode='bilinear', align_corners=False)
        logits_teacher = F.interpolate(logits_teacher, size=(h, w), mode='bilinear', align_corners=False)

        log_preds_student = F.log_softmax(logits_student / self.T, dim=1)
        preds_teacher = F.softmax(logits_teacher / self.T, dim=1)

        loss_pixelwise = self.kl_div(log_preds_student, preds_teacher) * (self.T ** 2)
        loss_pixelwise = torch.sum(loss_pixelwise, dim=1, keepdim=True)

        valid_mask = (target_masks != 255).float().unsqueeze(1)
        loss_kd = (loss_pixelwise * valid_mask).sum() / (valid_mask.sum() + 1e-6)
        return loss_kd


class BPKDLoss(nn.Module):
    """
    Boundary-Preserving KD (SoTA Logit)
    """
    def __init__(self, temperature=4.0, num_classes=150):
        super(BPKDLoss, self).__init__()
        self.T = temperature
        self.num_classes = num_classes
        self.kl_div = nn.KLDivLoss(reduction='none')

    def extract_boundary(self, masks):
        mask_clamp = torch.clamp(masks, min=0)
        clean_mask = torch.where(mask_clamp >= self.num_classes, torch.zeros_like(mask_clamp), mask_clamp)
        one_hot = F.one_hot(clean_mask, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        # Tìm biên bằng MaxPool2d để tránh lỗi nổ dải ô index của toán tử Conv2d Groups cũ
        max_pool = F.max_pool2d(one_hot, kernel_size=3, stride=1, padding=1)
        boundary = ((max_pool - one_hot).sum(dim=1, keepdim=True) > 0).float()

        # Loại bỏ các vùng nhãn ẩn (255) ra khỏi bản đồ biên cạnh
        valid_mask = (masks != 255).float().unsqueeze(1)
        return boundary * valid_mask

    def forward(self, logits_student, logits_teacher, target_masks):
        h, w = target_masks.shape[1], target_masks.shape[2]
        logits_student = F.interpolate(logits_student, size=(h, w), mode='bilinear', align_corners=False)
        logits_teacher = F.interpolate(logits_teacher, size=(h, w), mode='bilinear', align_corners=False)

        # Lấy bản đồ vùng biên bọc giáp bằng MaxPool
        boundary_mask = self.extract_boundary(target_masks)

        log_preds_student = F.log_softmax(logits_student / self.T, dim=1)
        preds_teacher = F.softmax(logits_teacher / self.T, dim=1)

        kl_div_pixelwise = self.kl_div(log_preds_student, preds_teacher) * (self.T ** 2)
        kl_div_pixelwise = torch.sum(kl_div_pixelwise, dim=1, keepdim=True)


        boundary_loss = (kl_div_pixelwise * boundary_mask).sum() / (boundary_mask.sum() + 1e-6)
        return boundary_loss

In [5]:
import os
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import SegformerImageProcessor
from torch.utils.data import DataLoader
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print(" Google Drive đã được kết nối từ trước!")

train_dataset = load_dataset("merve/scene_parse_150", split="train")
val_dataset = load_dataset("merve/scene_parse_150", split="validation")

# Khởi tạo bộ tiền xử lý hình ảnh của SegFormer
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
def collate_fn(batch):
    list_images = [x["image"].convert("RGB") for x in batch]
    list_labels = [x["annotation"] for x in batch]

    # Đưa qua bộ xử lý của Hugging Face để tạo Tensor hình ảnh và mặt nạ
    inputs = processor(images=list_images, segmentation_maps=list_labels, return_tensors="pt")

    # Lọc trùng nhãn:
    masks = inputs["labels"].long()

    NUM_CLASSES = 150
    masks = torch.where((masks >= NUM_CLASSES) & (masks != 255), torch.tensor(255), masks)
    masks = torch.where(masks < 0, torch.tensor(255), masks)

    inputs["labels"] = masks
    return inputs

BATCH_SIZE = 16

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    collate_fn=collate_fn
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    collate_fn=collate_fn
)

print(f"   - Tổng số lượng Batch huấn luyện (Train): {len(train_dataloader)}")
print(f"   - Tổng số lượng Batch kiểm thử (Val):     {len(val_dataloader)}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/212M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/89.1M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20210 [00:00<?, ? examples/s]

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

   - Tổng số lượng Batch huấn luyện (Train): 1264
   - Tổng số lượng Batch kiểm thử (Val):     125


/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


In [12]:
import os
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import evaluate
from transformers import SegformerForSemanticSegmentation

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

miou_metric = evaluate.load("mean_iou")

def train_kd_segformer_scientific(
    train_loader, val_loader, num_epochs=100, method="vanilla", num_classes=150,
    base_lr=6e-5, alpha=0.5, save_dir="/content/drive/MyDrive/official_kd_checkpoints",
    resume_path=None
):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Khởi tạo mô hình
    teacher = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-ade-640-640").to(device)
    teacher.eval()
    for param in teacher.parameters():
        param.requires_grad = False

    student = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512").to(device)

    # 2. Setup Loss, Optimizer, Scheduler
    criterion_ce = nn.CrossEntropyLoss(ignore_index=255)
    criterion_kd = VanillaKDLoss(temperature=4.0).to(device) if method == "vanilla" else BPKDLoss(temperature=4.0, num_classes=num_classes).to(device)

    optimizer = optim.AdamW(student.parameters(), lr=base_lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-7)

    scaler = torch.cuda.amp.GradScaler()
    mixed_precision_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

    start_epoch = 0
    best_val_miou = 0.0

    if resume_path and os.path.exists(resume_path):
        print(f"  Đang nạp lại trạng thái từ checkpoint: {resume_path}")
        checkpoint = torch.load(resume_path, map_location=device,weights_only=False)

        student.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_miou = checkpoint['best_miou']
        print(f" Sẽ chạy tiếp từ Epoch {start_epoch+1}, mIoU tốt nhất lịch sử: {best_val_miou:.4f}")

    print(f"=== HUẤN LUYỆN: {method.upper()} (Chạy từ Epoch {start_epoch+1}) ===")

    # 3. Vòng lặp Training chính thức
    for epoch in range(start_epoch, num_epochs):
        student.train()
        train_loss = 0.0
        current_lr = optimizer.param_groups[0]['lr']
        print(f"\n Epoch [{epoch+1}/{num_epochs}] | LR: {current_lr:.7f}")

        gc.collect()
        torch.cuda.empty_cache()

        for batch_idx, batch in enumerate(train_loader):
            images = batch['pixel_values'].to(device)
            masks = batch['labels'].to(device).long()

            masks = torch.where((masks >= num_classes) & (masks != 255), torch.tensor(255, device=device), masks)
            masks = torch.where(masks < 0, torch.tensor(255, device=device), masks)

            optimizer.zero_grad()

            with torch.autocast(device_type='cuda', dtype=mixed_precision_dtype):
                with torch.no_grad():
                    out_teacher = teacher(pixel_values=images).logits
                out_student = student(pixel_values=images).logits

            out_student = out_student.float()
            out_teacher = out_teacher.float()

            logits_upsampled = F.interpolate(out_student, size=masks.shape[1:], mode="bilinear", align_corners=False)
            loss_ce = criterion_ce(logits_upsampled, masks)
            loss_kd = criterion_kd(out_student, out_teacher, masks)

            total_loss = (1 - alpha) * loss_ce + alpha * loss_kd

            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += total_loss.item()

            if batch_idx % 100 == 0:
                print(f"Batch [{batch_idx}/{len(train_loader)}] | Loss: {total_loss.item():.4f} (CE: {loss_ce.item():.4f}, KD: {loss_kd.item():.4f})")

            del images, masks, out_student, out_teacher, logits_upsampled, total_loss

        avg_train_loss = train_loss / len(train_loader)
        scheduler.step()


        # 4. Validation mIoU định kỳ để tăng tốc
        VAL_EVERY_N_EPOCHS = 5
        avg_val_miou = 0.0

        if (epoch + 1) % VAL_EVERY_N_EPOCHS == 0 or (epoch + 1) == num_epochs:
            student.eval()
            print(f"--> 🔬 Đang tính mIoU trên tập Validation cho Epoch {epoch+1}...")

            with torch.no_grad():
                for batch in val_loader:
                    images = batch['pixel_values'].to(device)
                    masks = batch['labels'].to(device).long()

                    with torch.autocast(device_type='cuda', dtype=mixed_precision_dtype):
                        pred_seg = student(pixel_values=images).logits

                    pred_seg = F.interpolate(pred_seg.float(), size=masks.shape[1:], mode="bilinear", align_corners=False)
                    preds = pred_seg.argmax(dim=1)

                    miou_metric.add_batch(predictions=preds.detach().cpu().numpy(), references=masks.detach().cpu().numpy())
                    del images, masks, pred_seg, preds

            metrics = miou_metric.compute(num_labels=num_classes, ignore_index=255, reduce_labels=False)
            avg_val_miou = metrics["mean_iou"]
            print(f"📈 Kết quả định kỳ mIoU: {avg_val_miou:.4f}")
        else:
            print(f"⏩ Bỏ qua bước Validation ở Epoch {epoch+1} để tối ưu tốc độ phần cứng.")

        print("-" * 60)
        print(f" FINISH EPOCH {epoch+1} | Loss Train: {avg_train_loss:.4f} | Val mIoU gần nhất: {best_val_miou:.4f}")
        print("-" * 60)

        # ĐÓNG GÓI FULL STATE VÀ LƯU BACKUP LÊN DRIVE
        checkpoint_data = {
            'epoch': epoch,
            'model_state_dict': student.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_miou': max(avg_val_miou, best_val_miou)
        }

        # Lưu định kỳ mỗi 5 epoch lên Drive làm bảo hiểm
        if (epoch + 1) % 5 == 0:
            torch.save(checkpoint_data, os.path.join(save_dir, f"student_{method}_epoch_{epoch+1}.pth"))
            print(f" Đã sao lưu checkpoint an toàn cho Epoch {epoch+1} lên Drive.")

        # Cập nhật kỷ lục mIoU mới
        if avg_val_miou > best_val_miou:
            best_val_miou = avg_val_miou
            torch.save(checkpoint_data, os.path.join(save_dir, f"best_student_{method}_B0.pth"))
            print(f" Đã lưu Best Checkpoint với mIoU = {best_val_miou:.4f}")



train_kd_segformer_scientific(
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    num_epochs=100,
    method="vanilla",
    num_classes=150,
    base_lr=6e-5,
    save_dir="/content/drive/MyDrive/official_kd_checkpoints",
    resume_path="/content/drive/MyDrive/official_kd_checkpoints/student_vanilla_epoch_10.pth"
)

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

/tmp/ipykernel_3500/771499200.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


🔄 [CỨU HỘ] Đang nạp lại trạng thái từ checkpoint: /content/drive/MyDrive/official_kd_checkpoints/student_vanilla_epoch_10.pth
 Sẽ chạy tiếp từ Epoch 11, mIoU tốt nhất lịch sử: 0.0129
=== HUẤN LUYỆN: VANILLA (Chạy từ Epoch 11) ===

 Epoch [11/100] | LR: 0.0000585
Batch [0/1264] | Loss: 5.2018 (CE: 3.2331, KD: 7.1705)
Batch [100/1264] | Loss: 4.1004 (CE: 2.4568, KD: 5.7439)
Batch [200/1264] | Loss: 6.0249 (CE: 3.4681, KD: 8.5818)
Batch [300/1264] | Loss: 4.8043 (CE: 2.6291, KD: 6.9795)
Batch [400/1264] | Loss: 4.4508 (CE: 2.6202, KD: 6.2814)
Batch [500/1264] | Loss: 4.5610 (CE: 2.7962, KD: 6.3258)
Batch [600/1264] | Loss: 4.3807 (CE: 2.5555, KD: 6.2060)
Batch [700/1264] | Loss: 4.4288 (CE: 2.7947, KD: 6.0628)
Batch [800/1264] | Loss: 4.2178 (CE: 2.6576, KD: 5.7779)
Batch [900/1264] | Loss: 4.4821 (CE: 2.7947, KD: 6.1695)
Batch [1000/1264] | Loss: 5.8345 (CE: 3.8432, KD: 7.8257)
Batch [1100/1264] | Loss: 4.3888 (CE: 2.6494, KD: 6.1282)
Batch [1200/1264] | Loss: 4.4749 (CE: 2.7256, KD: 6.2

/usr/local/lib/python3.12/dist-packages/datasets/features/image.py:357: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")


📈 Kết quả định kỳ mIoU: 0.0138
------------------------------------------------------------
 FINISH EPOCH 15 | Loss Train: 4.4308 | Val mIoU gần nhất: 0.0129
------------------------------------------------------------
 Đã sao lưu checkpoint an toàn cho Epoch 15 lên Drive.
 Đã lưu Best Checkpoint với mIoU = 0.0138

 Epoch [16/100] | LR: 0.0000567
Batch [0/1264] | Loss: 4.8395 (CE: 3.1651, KD: 6.5140)
Batch [100/1264] | Loss: 4.6769 (CE: 2.9498, KD: 6.4039)
Batch [200/1264] | Loss: 4.7176 (CE: 2.8835, KD: 6.5517)
Batch [300/1264] | Loss: 4.7518 (CE: 2.8119, KD: 6.6917)
Batch [400/1264] | Loss: 4.4282 (CE: 2.5509, KD: 6.3055)
Batch [500/1264] | Loss: 4.1288 (CE: 2.6168, KD: 5.6407)
Batch [600/1264] | Loss: 4.6052 (CE: 2.7191, KD: 6.4913)
Batch [700/1264] | Loss: 4.5750 (CE: 2.8386, KD: 6.3113)
Batch [800/1264] | Loss: 4.3051 (CE: 2.6286, KD: 5.9816)
Batch [900/1264] | Loss: 4.0714 (CE: 2.4071, KD: 5.7356)
Batch [1000/1264] | Loss: 4.2192 (CE: 2.4957, KD: 5.9426)
Batch [1100/1264] | Loss:

KeyboardInterrupt: 